In [ ]:
import sys, subprocess, json, time
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    sys.path.insert(0, str(p))
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade",
                "transformers", "accelerate", "safetensors", "huggingface_hub"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn", "scipy", "bcubed==1.5"], check=True)

import torch, transformers
from huggingface_hub import HfApi
from transformers import AutoConfig, AutoProcessor, AutoModelForMultimodalLM
from project_utils import new_run, save_json, save_scores, read_scores, frozen, save_evaluation, record_environment
from semeval_task3.data import load_dataset

assert torch.cuda.is_available(), "Select an A100 runtime."
print("GPU:", torch.cuda.get_device_name(0), f"{torch.cuda.mem_get_info()[1] / 2**30:.0f} GiB")

MODEL_ID = "Qwen/Qwen3.8-27B"
MODEL_REVISION = HfApi().model_info(MODEL_ID).sha
config = AutoConfig.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
print("model_type:", config.model_type, "| revision:", MODEL_REVISION, "| transformers:", transformers.__version__)

RUN = new_run(PROJECT, "step04_qwen38_27b")
record_environment(PROJECT, RUN / "setup_manifest.json", model_id=MODEL_ID,
                   model_revision=MODEL_REVISION, gpu=torch.cuda.get_device_name(0))
print("Run:", RUN)

In [ ]:
t0 = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
tokenizer = processor.tokenizer
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, revision=MODEL_REVISION, dtype=torch.bfloat16,
    device_map={"": 0}, attn_implementation="sdpa").eval()
model.requires_grad_(False)
print(type(model).__name__, f"| GPU allocated {torch.cuda.memory_allocated() / 2**30:.1f} GiB",
      f"| {time.perf_counter() - t0:.0f}s")

In [ ]:
dev = load_dataset(PROJECT / "work/data/dev")  # inputs only, no labels
assert not any(u.span_flags for u in dev.usages)

SYSTEM = "You are an expert lexicographer of historical Swedish."
def messages(u):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": (
                f"Target word (lemma): {u.word}\n"
                f"Sense definition (Swedish): {u.definition}\n"
                f"Sentence from {u.year}: {u.marked_text()}\n\n"
                "Does the word marked with <t> and </t> in the sentence express "
                "the sense given by the definition? Answer only Yes or No.")}]

yes_ids, no_ids = tokenizer.encode("Yes", add_special_tokens=False), tokenizer.encode("No", add_special_tokens=False)
print("Yes ids:", yes_ids, "| No ids:", no_ids)
assert len(yes_ids) == len(no_ids) == 1, "Not single tokens: stop and paste this output."
YES, NO = yes_ids[0], no_ids[0]

def encode(batch_msgs):
    return processor.apply_chat_template(
        batch_msgs, tokenize=True, return_dict=True, return_tensors="pt",
        add_generation_prompt=True, enable_thinking=False,
        processor_kwargs={"padding": True, "truncation": False}).to("cuda")

first = dev.usages[0]
second = next(u for u in dev.usages if u.group != first.group)
for u in [first, second]:
    inputs = encode([messages(u)])
    with torch.inference_mode():
        logits = model(**inputs, use_cache=False, logits_to_keep=1).logits[0, -1].float()
        gen = model.generate(**inputs, max_new_tokens=8, do_sample=False)
    answer = tokenizer.decode(gen[0, inputs["input_ids"].shape[-1]:], skip_special_tokens=False)
    p = torch.softmax(logits, -1)
    top = [(tokenizer.decode([i]), round(v, 4)) for v, i in zip(*[t.tolist() for t in p.topk(5)])]
    print(f"\n{u.word}: answer={answer!r}  margin={(logits[YES] - logits[NO]).item():.3f}  "
          f"Yes/No mass={(p[YES] + p[NO]).item():.4f}\n  top tokens: {top}")
    assert answer.strip().startswith(("Yes", "No")), "Model is not answering Yes/No directly. STOP and paste output."
print("\nSMOKE TEST PASSED")

In [ ]:
tokenizer.padding_side = "left"
processor.tokenizer.padding_side = "left"
scores, low_mass, t0, B = {}, 0, time.perf_counter(), 4
torch.cuda.reset_peak_memory_stats()
with torch.inference_mode():
    for i in range(0, len(dev.usages), B):
        batch = dev.usages[i:i + B]
        inputs = encode([messages(u) for u in batch])
        assert inputs["attention_mask"][:, -1].all().item(), "Padding is not on the left."
        logits = model(**inputs, use_cache=False, logits_to_keep=1).logits[:, -1, :].float()
        p = torch.softmax(logits, -1)
        low_mass += int(((p[:, YES] + p[:, NO]) < 0.9).sum())
        scores.update({u.key: v for u, v in zip(batch, (logits[:, YES] - logits[:, NO]).tolist())})
        if i % 400 == 0:
            print(f"{i}/{len(dev.usages)}  {time.perf_counter() - t0:.0f}s")
sec = time.perf_counter() - t0
save_scores(RUN / "qwen38_raw_scores.json", scores)
record_environment(PROJECT, RUN / "inference_manifest.json", model_id=MODEL_ID, model_revision=MODEL_REVISION,
                   prompt="identical to Gemma v1", thinking=False, batch_size=B, seconds=sec,
                   peak_gpu_gib=torch.cuda.max_memory_allocated() / 2**30, low_yes_no_mass=low_mass)
print(f"Done: {len(scores)} scores in {sec:.0f}s ({1000 * sec / len(scores):.0f}s per 1,000); "
      f"Yes/No mass < 0.9: {low_mass}; range {min(scores.values()):.2f} to {max(scores.values()):.2f}")

In [ ]:
import numpy as np
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap

eval_dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = eval_dev.require_gold()
GEMMA_SCORES = PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json"
GEMMA_ZIP = PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a/gemma_raw.zip"
gemma_v1 = score(gold, read_submission(GEMMA_ZIP, eval_dev.keys))
assert abs(gemma_v1["average"] - 0.6782392862783919) < 1e-10

GRID = tuple([-1000.0] + [round(float(x), 2) for x in np.arange(-40, 40.001, 0.25)] + [1000.0])
qwen = read_scores(RUN / "qwen38_raw_scores.json", eval_dev)
assert all(-40 <= v <= 40 for v in qwen.values())

def evaluate(name, sc):
    res = nested_word_cv(eval_dev, [Candidate(name, frozen(sc), GRID)])
    save_evaluation(RUN, name, res, eval_dev, PROJECT, official=True)
    pair = word_bootstrap(res["report"], other=gemma_v1)
    gw = {r["word"]: r["f1"] for r in gemma_v1["per_word"]}
    wins = 0
    print(f"\n{'word':<10}{'Gemma':>7}{name[:8]:>9}{'diff':>8}")
    for r in res["report"]["per_word"]:
        d = r["f1"] - gw[r["word"]]; wins += d > 0
        print(f"{r['word']:<10}{gw[r['word']]:>7.3f}{r['f1']:>9.3f}{d:>+8.3f}")
    print(f"{name} minus Gemma v1: {pair['estimate']:+.4f} [{pair['lower']:+.4f}, {pair['upper']:+.4f}]; "
          f"better on {wins}/10 | thresholds {[float(s['threshold']) for s in res['selections']]}")
    return res, pair, wins

qwen_res, qwen_pair, qwen_wins = evaluate("qwen38_27b_raw", qwen)

In [ ]:
gemma = read_scores(GEMMA_SCORES, eval_dev)
ens = {k: (gemma[k] + qwen[k]) / 2 for k in gemma}   # average of Yes-No log-odds
save_scores(RUN / "ensemble_gemma_qwen38_scores.json", ens)
ens_res, ens_pair, ens_wins = evaluate("ensemble_mean_margin", ens)

adopt = ens_pair["estimate"] >= 0.02 and ens_wins >= 6
save_json(RUN / "ensemble_decision.json", {
    "rule": "adopt ensemble if paired gain vs Gemma v1 >= +0.02 and better on >= 6/10 words (set before results)",
    "ensemble": "mean of Gemma v1 and Qwen3.8-27B Yes-No margins, identical prompt",
    "qwen_alone_vs_gemma": qwen_pair, "ensemble_vs_gemma": ens_pair,
    "ensemble_words_better": ens_wins, "adopt": adopt})
print("\nDECISION:", "ADOPT ENSEMBLE" if adopt else "KEEP GEMMA v1 ALONE")

In [ ]:
import json
import csv
import sys
import hashlib
import zipfile
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
sys.path.insert(0, str(PROJECT / "src"))

from semeval_task3.evaluation import word_bootstrap

RUN = PROJECT / (
    "outputs/runs/step04_qwen38_27b/"
    "20261105T193650Z_24c46b6f"
)
GEMMA_RUN = PROJECT / (
    "outputs/runs/step04_gemma4_evaluation/"
    "20261105T170910Z_b3fc303a"
)

OUT = RUN / "results_summary"
OUT.mkdir(parents=True, exist_ok=True)

def read_json(path):
    assert path.is_file(), f"Missing saved result: {path}"
    return json.loads(path.read_text(encoding="utf-8"))

result_files = {
    "Gemma v1": GEMMA_RUN / "gemma_raw.json",
    "Qwen3.8-27B": RUN / "qwen38_27b_raw.json",
    "Gemma + Qwen mean-margin ensemble": RUN / "ensemble_mean_margin.json",
}
results = {name: read_json(path) for name, path in result_files.items()}
reference = results["Gemma v1"]["report"]

rows = []
per_word_rows = []

for name, result in results.items():
    report = result["report"]
    ci = result["confidence_interval"]
    delta = word_bootstrap(report, other=reference)

    rows.append({
        "system": name,
        "mean_per_word_f1": report["average"],
        "f1_ci_lower": ci["lower"],
        "f1_ci_upper": ci["upper"],
        "difference_vs_gemma": delta["estimate"],
        "difference_ci_lower": delta["lower"],
        "difference_ci_upper": delta["upper"],
    })

    for word_result in report["per_word"]:
        per_word_rows.append({"system": name, **word_result})

def write_csv(path, records):
    with path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(records[0]))
        writer.writeheader()
        writer.writerows(records)

write_csv(OUT / "results.csv", rows)
write_csv(OUT / "per_word_results.csv", per_word_rows)

summary = {
    "dataset": "Swedish development data",
    "examples": sum(r["n"] for r in reference["per_word"]),
    "words": len(reference["per_word"]),
    "metric": "Positive-class F1 averaged over words",
    "evaluation": "Nested leave-one-word-out threshold selection",
    "results": rows,
    "current_reference": "Gemma v1",
    "decision": "Retain original Gemma; neither candidate improves observed mean F1.",
    "limitations": [
        "Development results, not official test or leaderboard scores.",
        "Bootstrap intervals do not account for repeated experiment selection.",
        "The ensemble averages uncalibrated margins from different models.",
    ],
    "source_files": {name: str(path) for name, path in result_files.items()},
}

(OUT / "summary.json").write_text(
    json.dumps(summary, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8"
)

lines = [
    "# Step 4: Qwen3.8-27B and ensemble results",
    "",
    "Evaluation: 1,983 Swedish development examples across 10 words.",
    "Metric: mean per-word F1; thresholds selected with nested word CV.",
    "",
    "| System | F1 | Difference vs Gemma |",
    "|---|---:|---:|",
]
for row in rows:
    lines.append(
        f"| {row['system']} | "
        f"{100 * row['mean_per_word_f1']:.2f}% | "
        f"{100 * row['difference_vs_gemma']:+.2f} points |"
    )

lines += [
    "",
    "Decision: retain original Gemma v1.",
    "",
    "Neither candidate improved observed mean F1. Both paired difference",
    "intervals include zero. These are exploratory development comparisons.",
    "",
    "The raw-margin ensemble does not establish whether a calibrated",
    "ensemble would help.",
]
(OUT / "RESULTS.md").write_text("\n".join(lines) + "\n", encoding="utf-8")

# Bundle this run's saved scores, predictions, metrics, and manifests.
# Include the original Gemma evaluation as the comparison reference.
backup = OUT / "qwen38_results_backup.zip"
source_files = [
    p for p in RUN.iterdir()
    if p.is_file() and p.suffix in {".json", ".zip", ".csv", ".md"}
]
reference_files = [
    GEMMA_RUN / "gemma_raw.json",
    GEMMA_RUN / "gemma_raw.zip",
    GEMMA_RUN / "configuration.json",
]
summary_files = [
    OUT / "results.csv",
    OUT / "per_word_results.csv",
    OUT / "summary.json",
    OUT / "RESULTS.md",
]

manifest = {}
with zipfile.ZipFile(backup, "w", zipfile.ZIP_DEFLATED) as archive:
    for prefix, paths in [
        ("qwen_run", source_files),
        ("gemma_reference", reference_files),
        ("summary", summary_files),
    ]:
        for path in paths:
            assert path.is_file(), f"Missing backup file: {path}"
            archive_name = f"{prefix}/{path.name}"
            content = path.read_bytes()
            archive.writestr(archive_name, content)
            manifest[archive_name] = hashlib.sha256(content).hexdigest()

    archive.writestr(
        "checksums.json",
        json.dumps(manifest, indent=2) + "\n"
    )

with zipfile.ZipFile(backup) as archive:
    assert archive.testzip() is None, "ZIP verification failed."

print("RESULTS SAVED — ZIP verification PASS")
for row in rows:
    print(f"{row['system']}: F1={row['mean_per_word_f1']:.6f}")
print("\nReport:", OUT / "RESULTS.md")
print("Table:", OUT / "results.csv")
print("Backup:", backup)

In [ ]:
import sys, numpy as np
from pathlib import Path
from google.colab import drive
drive.mount("/content/drive")
PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for p in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(p) not in sys.path: sys.path.insert(0, str(p))
import subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "scikit-learn", "scipy", "bcubed==1.5"], check=True)

from project_utils import read_scores, frozen, save_evaluation, save_json, save_scores, new_run, record_environment
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import score, read_submission, word_bootstrap

dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = dev.require_gold()
g = read_scores(PROJECT / "outputs/runs/step04_gemma4_31b/20261105T164154Z_48096e67/gemma_raw_scores.json", dev)
qpaths = sorted((PROJECT / "outputs/runs/step04_qwen38_27b").glob("*/qwen38_raw_scores.json"))
assert len(qpaths) == 1, f"Expected one Qwen3.8 score file, found: {qpaths}"
q = read_scores(qpaths[0], dev)

# 1. Label-free calibration: put both models on the same scale (mean 0, std 1 over all inputs)
def standardize(s):
    v = np.array(list(s.values())); m, sd = v.mean(), v.std()
    return {k: (x - m) / sd for k, x in s.items()}
gz, qz = standardize(g), standardize(q)

# 2. One candidate per weight; nested CV picks weight AND threshold inside training folds only
WEIGHTS = [round(w, 1) for w in np.arange(0.0, 1.0001, 0.1)]   # 1.0 = Gemma alone, 0.0 = Qwen alone
GRID = tuple([-1000.0] + [round(float(x), 3) for x in np.arange(-6, 6.0001, 0.025)] + [1000.0])
RUN = new_run(PROJECT, "step04_calibrated_ensemble")
candidates = []
for w in WEIGHTS:
    comb = {k: w * gz[k] + (1 - w) * qz[k] for k in gz}
    assert all(-6 <= v <= 6 for v in comb.values()), f"w={w}: scores outside grid"
    save_scores(RUN / f"combined_w{w:.1f}.json", comb)
    candidates.append(Candidate(f"w_gemma={w:.1f}", frozen(comb), GRID))

res = nested_word_cv(dev, candidates)
save_evaluation(RUN, "calibrated_ensemble", res, dev, PROJECT, official=True)

# 3. Paired comparison against Gemma v1
v1 = score(gold, read_submission(PROJECT / "outputs/runs/step04_gemma4_evaluation/20261105T170910Z_b3fc303a/gemma_raw.zip", dev.keys))
assert abs(v1["average"] - 0.6782392862783919) < 1e-10
pair = word_bootstrap(res["report"], other=v1)
gw = {r["word"]: r["f1"] for r in v1["per_word"]}
sel = {s["test_group"][1]: s for s in res["selections"]}
wins = 0
print(f"{'word':<10}{'Gemma':>7}{'Ens':>7}{'diff':>8}   chosen weight / threshold")
for r in res["report"]["per_word"]:
    d = r["f1"] - gw[r["word"]]; wins += d > 0
    s = sel[r["word"]]
    print(f"{r['word']:<10}{gw[r['word']]:>7.3f}{r['f1']:>7.3f}{d:>+8.3f}   {s['candidate']} / {float(s['threshold']):.3f}")

adopt = pair["estimate"] >= 0.02 and wins >= 6
print(f"\nCalibrated ensemble F1: {res['report']['average']:.4f}  (Gemma v1: {v1['average']:.4f})")
print(f"Ensemble minus Gemma v1: {pair['estimate']:+.4f} [{pair['lower']:+.4f}, {pair['upper']:+.4f}]; better on {wins}/10")
print("DECISION:", "ADOPT calibrated ensemble" if adopt else "KEEP GEMMA v1")
save_json(RUN / "decision.json", {
    "rule": "adopt if paired gain vs Gemma v1 >= +0.02 and better on >= 6/10 words (set before results)",
    "method": "global label-free standardization; weight in 0.0..1.0 step 0.1 and threshold chosen by inner folds only",
    "paired_vs_gemma_v1": pair, "words_better": wins, "adopt": adopt,
    "chosen_per_fold": {k: v["candidate"] for k, v in sel.items()}})
record_environment(PROJECT, RUN / "manifest.json", purpose="Calibrated Gemma+Qwen3.8 ensemble, nested weight selection",
                   qwen_scores=str(qpaths[0]), new_model_inference=False)

In [ ]:
import sys
import json
import numpy as np
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

PROJECT = Path("/content/drive/MyDrive/semeval2027/semeval2027_colab")
for folder in [PROJECT / "src", PROJECT / "notebooks"]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

from project_utils import (
    read_scores, frozen, save_evaluation, save_json,
    new_run, record_environment, digest
)
from semeval_task3.data import load_dataset
from semeval_task3.cv import Candidate, nested_word_cv
from semeval_task3.evaluation import (
    score, read_submission, word_bootstrap
)

# 1. Load the dataset and previously saved model scores.
dev = load_dataset(PROJECT / "work/data/dev", with_gold=True)
gold = dev.require_gold()

gemma_path = PROJECT / (
    "outputs/runs/step04_gemma4_31b/"
    "20261105T164154Z_48096e67/gemma_raw_scores.json"
)
qwen_path = PROJECT / (
    "outputs/runs/step04_qwen38_27b/"
    "20261105T193650Z_24c46b6f/qwen38_raw_scores.json"
)
reference_dir = PROJECT / (
    "outputs/runs/step04_gemma4_evaluation/"
    "20261105T170910Z_b3fc303a"
)

g = read_scores(gemma_path, dev)
q = read_scores(qwen_path, dev)

reference_predictions = read_submission(
    reference_dir / "gemma_raw.zip", set(dev.keys)
)
reference_report = score(gold, reference_predictions)

assert abs(reference_report["average"] - 0.6782392862783919) < 1e-10
print(f"Original Gemma reference: {reference_report['average']:.6f}")

# Reuse the original Gemma threshold grid exactly.
reference_config = json.loads(
    (reference_dir / "configuration.json").read_text(encoding="utf-8")
)
RAW_GRID = tuple(float(t) for t in reference_config["raw_grid"])

WEIGHTS = tuple(i / 10 for i in range(11))
ENSEMBLE_GRID = (
    -1000.0,
    *(round(i * 0.025, 3) for i in range(-240, 241)),
    1000.0,
)

# 2. Each fit computes normalization from ONLY its training usages.
# During inner CV this means 8 words; during outer refit, 9 words.
# Neither held-out inputs nor held-out labels determine these statistics.
def ensemble_trainer(weight):
    def fit(train_usages, train_labels):
        # Labels are deliberately unused for normalization.
        train_keys = [u.key for u in train_usages]

        gv = np.asarray([g[k] for k in train_keys], dtype=float)
        qv = np.asarray([q[k] for k in train_keys], dtype=float)

        gm, gs = float(gv.mean()), float(gv.std())
        qm, qs = float(qv.mean()), float(qv.std())

        assert gs > 1e-8 and qs > 1e-8, "Degenerate training score scale."

        def predict(test_usages):
            values = {
                u.key: float(
                    weight * ((g[u.key] - gm) / gs)
                    + (1.0 - weight) * ((q[u.key] - qm) / qs)
                )
                for u in test_usages
            }
            assert all(np.isfinite(v) for v in values.values())
            # Ensure sentinel thresholds cover all-positive/all-negative.
            assert all(-1000.0 < v < 1000.0 for v in values.values())
            return values

        return predict

    return fit

# Prefix gives original Gemma preference when inner scores tie exactly.
candidates = [
    Candidate("00_original_gemma", frozen(g), RAW_GRID)
]
candidates += [
    Candidate(
        f"ensemble_gemma_weight_{weight:.1f}",
        ensemble_trainer(weight),
        ENSEMBLE_GRID
    )
    for weight in WEIGHTS
]

VERIFY_RUN = new_run(PROJECT, "step04_ensemble_fold_verification")

save_json(VERIFY_RUN / "configuration.json", {
    "gemma_scores": str(gemma_path),
    "gemma_scores_sha256": digest(gemma_path),
    "qwen_scores": str(qwen_path),
    "qwen_scores_sha256": digest(qwen_path),
    "reference_predictions_sha256": digest(reference_dir / "gemma_raw.zip"),
    "weights": list(WEIGHTS),
    "original_threshold_grid": list(RAW_GRID),
    "ensemble_threshold_grid": list(ENSEMBLE_GRID),
    "normalization": (
        "Separate mean and population standard deviation for each model, "
        "computed over training usages only, refitted in every inner and outer fold."
    ),
    "selection": (
        "Original Gemma or standardized mixture, plus threshold, "
        "selected using inner held-out-word F1."
    ),
    "limitation": (
        "Exploratory development experiment following previous development "
        "comparisons. Bootstrap intervals do not include experiment-selection uncertainty."
    )
})

# 3. Confirm the exact original configuration is reproducible.
print("\nChecking original Gemma reproduction...")
baseline_check = nested_word_cv(dev, [candidates[0]])

assert baseline_check["predictions"] == reference_predictions, (
    "Original predictions were not reproduced. Stop before comparing."
)
print("Original Gemma predictions reproduced: PASS")

# 4. Evaluate the complete selection procedure.
print("\nSelecting candidate, weight, and threshold inside inner folds...")
print("This is CPU evaluation; it may take several minutes.")

verified_result = nested_word_cv(dev, candidates)

save_evaluation(
    VERIFY_RUN,
    "fold_fitted_selection",
    verified_result,
    dev,
    PROJECT,
    official=True
)
print("Official scorer agreement: PASS")

# 5. Paired comparison against original Gemma.
paired = word_bootstrap(
    verified_result["report"], other=reference_report
)
save_json(VERIFY_RUN / "paired_vs_original_gemma.json", paired)

old_words = {
    (r["language"], r["word"]): r
    for r in reference_report["per_word"]
}
selections = {
    tuple(s["test_group"]): s
    for s in verified_result["selections"]
}

print(
    f"\n{'Word':12} {'Gemma':>8} {'Verified':>9} "
    f"{'Change':>9}  Selected candidate / threshold"
)

for row in verified_result["report"]["per_word"]:
    key = (row["language"], row["word"])
    old = old_words[key]
    selected = selections[key]
    print(
        f"{row['word']:12} {old['f1']:8.4f} {row['f1']:9.4f} "
        f"{row['f1'] - old['f1']:+9.4f}  "
        f"{selected['candidate']} / {selected['threshold']:.3f}"
    )

print(f"\nOriginal Gemma F1: {reference_report['average']:.6f}")
print(f"Verified procedure F1: {verified_result['report']['average']:.6f}")
print(
    f"Difference: {100 * paired['estimate']:+.2f} F1 points\n"
    f"95% paired interval: "
    f"[{100 * paired['lower']:+.2f}, {100 * paired['upper']:+.2f}] points"
)

_ = record_environment(
    PROJECT,
    VERIFY_RUN / "manifest.json",
    purpose="Training-fold normalization and nested candidate selection",
    new_model_inference=False
)

print("\nVERIFICATION COMPLETE")
print("Saved:", VERIFY_RUN)
print("Original Gemma files remain unchanged.")